# Hateful Meme Detection — GPU Training
Before running: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
!git clone https://github.com/jgodwin31/AIDL-final-project.git
%cd AIDL-final-project

Cloning into 'AIDL-final-project'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 56 (delta 11), reused 56 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 1.12 MiB | 13.86 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/AIDL-final-project


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
!python download_dataset.py

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Rate limited. Waiting 21.0s before retry [Retry 1/5].
Fetching ... files: 2992it [03:36, 15.41it/s]HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/limjiayi/hateful_memes_expanded/resolve/9fe4913f7ee6c9a1ddd8f26e5eacbfb3d263782b/img/25061.png
Rate limited. Waiting 21.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/limjiayi/hateful_memes_expanded/resolve/9fe4913f7ee6c9a1ddd8f26e5eacbfb3d263782b/img/25067.png
Rate limited. Waiting 21.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/limjiayi/hateful_memes_expanded/resolve/9fe4913f7ee6c9a1ddd8f26e5eacbfb3d263782b/img/25068.png
Rate limited. Wa

In [ ]:
import yaml
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["training"]["device"] = "cuda"
cfg["training"]["num_epochs"] = 5
cfg["training"]["batch_size"] = 32
cfg["training"]["num_workers"] = 2
with open("config/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print("Config updated — device: cuda")

Config updated — device: cuda


## 1. Text Model (BERT)

In [ ]:
!python train_text.py 2>&1 | tee logs/text_colab.log

Using device: cuda
Freeze encoder: False

Loading data...
Generating test split: 100%|██████████| 3000/3000 [00:00<00:00, 671375.09 examples/s]
  Train batches: 403  |  Val batches: 33

Building model...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3813.87it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture

In [ ]:
import glob, os
best_text = sorted(glob.glob("checkpoints/text_best*.pt"))[-1]
print("Best:", best_text)
!python evaluate.py --model text --checkpoint {best_text} --split validation
print("--- TEST ---")
!python evaluate.py --model text --checkpoint {best_text} --split test

Best: checkpoints/text_best_epoch2.pt
Model type : text
Checkpoint : checkpoints/text_best_epoch2.pt
Split      : validation
Device     : cuda

Loading weights: 100% 199/199 [00:00<00:00, 528.89it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[Checkpoint] Loaded ← checkpoints/

## 2. Image Model (ResNet18)

In [ ]:
!python train_image.py 2>&1 | tee logs/image_colab.log

Using device: cuda
Freeze encoder: False  |  Augmentation: True

Loading data...
  Train batches: 403  |  Val batches: 33

Building model...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]
  Trainable parameters: 11,308,354

Starting training...

=== New run started at 2026-05-10 03:08:17 ===

--- Epoch 1 ---
[Epoch 1 | Step 50]  loss=0.6300  
[Epoch 1 | Step 100]  loss=0.6514  
[Epoch 1 | Step 150]  loss=0.5605  
[Epoch 1 | Step 200]  loss=0.6415  
[Epoch 1 | Step 250]  loss=0.5904  
[Epoch 1 | Step 300]  loss=0.4067  
[Epoch 1 | Step 350]  loss=0.4423  
[Epoch 1 | Step 400]  loss=0.5836  

[VAL Metrics]
  AUROC    : 0.4620
  F1       : 0.1331
  Accuracy : 0.5365
  Confusion Matrix:
[[521  72]
 [410  37]]
[VAL @ Epoch 1]  train_loss=0.5802  val_loss=0.7336  auroc=0.4620  f1=0.1331  accuracy=0.5365
[Checkpoint] Saved → checkpoints/image_epoch1.pt
[Che

In [ ]:
best_image = sorted(glob.glob("checkpoints/image_best*.pt"))[-1]
print("Best:", best_image)
!python evaluate.py --model image --checkpoint {best_image} --split validation
print("--- TEST ---")
!python evaluate.py --model image --checkpoint {best_image} --split test

Best: checkpoints/image_best_epoch2.pt
Model type : image
Checkpoint : checkpoints/image_best_epoch2.pt
Split      : validation
Device     : cuda

[Checkpoint] Loaded ← checkpoints/image_best_epoch2.pt  (epoch 2)

[VALIDATION Metrics]
  AUROC    : 0.5087
  F1       : 0.0757
  Accuracy : 0.5538
  Confusion Matrix:
[[557  36]
 [428  19]]
--- TEST ---
Model type : image
Checkpoint : checkpoints/image_best_epoch2.pt
Split      : test
Device     : cuda

[Checkpoint] Loaded ← checkpoints/image_best_epoch2.pt  (epoch 2)

[TEST Metrics]
  AUROC    : 0.5386
  F1       : 0.1275
  Accuracy : 0.5847
  Confusion Matrix:
[[1663   97]
 [1149   91]]


## 3. Fusion Model (BERT + ResNet18)

In [ ]:
!python train_fusion.py 2>&1 | tee logs/fusion_colab.log

Using device: cuda
Freeze encoders: False

Loading data...
  Train batches: 403  |  Val batches: 33

Building model...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 640.98it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Trainable / Total parameters: 121,446,466 / 1

In [ ]:
best_fusion = sorted(glob.glob("checkpoints/fusion_best*.pt"))[-1]
print("Best:", best_fusion)
!python evaluate.py --model fusion --checkpoint {best_fusion} --split validation
print("--- TEST ---")
!python evaluate.py --model fusion --checkpoint {best_fusion} --split test

Best: checkpoints/fusion_best_epoch3.pt
Model type : fusion
Checkpoint : checkpoints/fusion_best_epoch3.pt
Split      : validation
Device     : cuda

Loading weights: 100% 199/199 [00:00<00:00, 572.42it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[Checkpoint] Loaded ← checkp

## 4. CLIP Model (frozen encoder)

In [ ]:
!python train_clip.py 2>&1 | tee logs/clip_colab.log

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Using device: cuda
Freeze CLIP encoder: True

Loading data...
  Train batches: 403  |  Val batches: 33

Building model...
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2769.66it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Trainable / Total parameters: 262,914 / 151,540

In [ ]:
best_clip = sorted(glob.glob("checkpoints/clip_best*.pt"))[-1]
print("Best:", best_clip)
!python evaluate.py --model clip --checkpoint {best_clip} --split validation
print("--- TEST ---")
!python evaluate.py --model clip --checkpoint {best_clip} --split test

Best: checkpoints/clip_best_epoch5.pt
Model type : clip
Checkpoint : checkpoints/clip_best_epoch5.pt
Split      : validation
Device     : cuda

Loading weights: 100% 398/398 [00:00<00:00, 923.18it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
[Checkpoint] Loaded ← checkpoints/clip_b

## 5. Print all logs

In [ ]:
for log in ["logs/text_colab.log","logs/image_colab.log",
            "logs/fusion_colab.log","logs/clip_colab.log"]:
    if os.path.exists(log):
        print("=" * 60)
        print(log)
        print("=" * 60)
        print(open(log).read())

logs/text_colab.log
Using device: cuda
Freeze encoder: False

Loading data...

Generating train split: 100%|██████████| 12887/12887 [00:00<00:00, 448106.88 examples/s]

Generating validation split: 100%|██████████| 1040/1040 [00:00<00:00, 361607.91 examples/s]

Generating test split: 100%|██████████| 3000/3000 [00:00<00:00, 671375.09 examples/s]
  Train batches: 403  |  Val batches: 33

Building model...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3813.87it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
c

In [ ]:
from google.colab import files
import zipfile
with zipfile.ZipFile("best_checkpoints.zip", "w") as z:
    for pattern in ["text_best","image_best","fusion_best","clip_best"]:
        matches = sorted(glob.glob(f"checkpoints/{pattern}*.pt"))
        if matches:
            z.write(matches[-1])
            print("Zipped:", matches[-1])
files.download("best_checkpoints.zip")

Zipped: checkpoints/text_best_epoch2.pt
Zipped: checkpoints/image_best_epoch2.pt
Zipped: checkpoints/fusion_best_epoch3.pt
Zipped: checkpoints/clip_best_epoch5.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>